In [11]:
!pip install tensorflow opencv-python mediapipe fastapi uvicorn pillow nest-asyncio numpy
print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# STEP 1: Setup Folders
import os
home = os.path.expanduser("~")
BASE = os.path.join(home, "Desktop", "GesturaAI_ANN")
folders = ["backend", "frontend", "report"]
os.makedirs(BASE, exist_ok=True)
for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print(f"✅ Folders ready at: {BASE}")

✅ Folders ready at: C:\Users\SHAHID BALOCH\Desktop\GesturaAI_ANN


In [2]:
# STEP 2: Hand & Face Test
import cv2
import mediapipe as mp
import time
from IPython.display import display, Image, clear_output
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
face_mesh = mp_face.FaceMesh(max_num_faces=1, refine_landmarks=False, min_detection_confidence=0.5)
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
if not cap.isOpened():
    print("❌ Webcam not found!")
else:
    print("✅ Webcam opened. Show hands and face.")
    prev_t = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            hres = hands.process(rgb)
            if hres.multi_hand_landmarks:
                for lm in hres.multi_hand_landmarks:
                    mp_draw.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS,
                                           mp_styles.get_default_hand_landmarks_style(),
                                           mp_styles.get_default_hand_connections_style())

            fres = face_mesh.process(rgb)
            if fres.multi_face_landmarks:
                for lm in fres.multi_face_landmarks:
                    mp_draw.draw_landmarks(frame, lm, mp_face.FACEMESH_TESSELATION,
                                           landmark_drawing_spec=None,
                                           connection_drawing_spec=mp_styles.get_default_face_mesh_tesselation_style())

            now = time.time()
            fps = 1/(now-prev_t) if prev_t else 0
            prev_t = now
            cv2.putText(frame, f"FPt: {int(fps)}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
            _, jpg = cv2.imencode('.jpg', frame)
            display(Image(data=jpg.tobytes()))
            clear_output(wait=True)
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
        hands.close()
        face_mesh.close()
        print("✅ Webcam stopped.")

✅ Webcam stopped.


In [3]:
# STEP 3: Collect Data
import cv2
import mediapipe as mp
import csv
import os
import time
from IPython.display import display, Image, clear_output
home = os.path.expanduser("~")
BASE = os.path.join(home, "Desktop", "GesturaAI_ANN")
csv_path = os.path.join(BASE, "dataset.csv")
GESTURES = ["thumbs_up", "peace", "fist", "ok", "palm"]
SAMPLES_PER_GESTURE = 100
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.6)
with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    header = ['label'] + [f'{axis}{i}' for i in range(21) for axis in ['x', 'y']]
    writer.writerow(header)
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
if not cap.isOpened():
    print("❌ Webcam not found!")
else:
    print("✅ Webcam opened. Get ready!")
    try:
        for gesture in GESTURES:
            print(f"\n👉 Please show: {gesture.upper()} (Starting in 3 sec...)")
            time.sleep(3) 
            count = 0
            while count < SAMPLES_PER_GESTURE:
                ret, frame = cap.read()
                if not ret: break
                frame = cv2.flip(frame, 1)
                h, w = frame.shape[:2]
                x1, y1, x2, y2 = w//2 - 150, h//2 - 150, w//2 + 150, h//2 + 150
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = hands.process(rgb)
    
                if results.multi_hand_landmarks:
                    for hand_landmarks in results.multi_hand_landmarks:
                        mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                        
                        row_data = [gesture]
                        for lm in hand_landmarks.landmark:
                            row_data.extend([round(lm.x, 4), round(lm.y, 4)])
                        
                        with open(csv_path, mode='a', newline='') as f:
                            writer = csv.writer(f)
                            writer.writerow(row_data)
                        count += 1
                        
                cv2.putText(frame, f"{gesture}: {count}/{SAMPLES_PER_GESTURE}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                _, jpg = cv2.imencode('.jpg', frame)
                display(Image(data=jpg.tobytes()))
                clear_output(wait=True)
                time.sleep(0.01)
                
            print(f"✅ Data collected for: {gesture}")
        print("\n🎉 All data collected successfully!")
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        cap.release()
        hands.close()

✅ Data collected for: palm

🎉 All data collected successfully!


In [4]:
# STEP 4: Train 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models
import os
home = os.path.expanduser("~")
BASE = os.path.join(home, "Desktop", "GesturaAI_ANN")
csv_path = os.path.join(BASE, "dataset.csv")

model_path = os.path.join(BASE, "ann_model.h5")
df = pd.read_csv(csv_path)
X = df.drop('label', axis=1).values
y_labels = df['label'].values
encoder = LabelEncoder()
y = encoder.fit_transform(y_labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(np.unique(y)), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("🚀 Training Started...")
model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), batch_size=32)
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n✅ ANN Model Accuracy: {accuracy * 100:.2f}%")
model.save(model_path)
np.save(os.path.join(BASE, "classes.npy"), encoder.classes_)
print("✅ Model successfully saved!")

🚀 Training Started...
Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.2125 - loss: 1.6142 - val_accuracy: 0.5700 - val_loss: 1.5462
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2525 - loss: 1.5441 - val_accuracy: 0.4600 - val_loss: 1.4532
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4900 - loss: 1.4382 - val_accuracy: 0.7200 - val_loss: 1.3417
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5725 - loss: 1.3485 - val_accuracy: 0.6900 - val_loss: 1.2522
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5750 - loss: 1.2588 - val_accuracy: 0.7700 - val_loss: 1.1184
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6050 - loss: 1.1485 - val_accuracy: 0.7500 - val_loss: 1.0544
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6175 - loss: 1.0685 - val_accuracy: 0.7200 - val_loss: 0.9784
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6550 - loss: 0.9969 - va


✅ ANN Model Accuracy: 83.00%
✅ Model successfully saved!


In [5]:
# STEP 5: Backend Server 
import threading
import uvicorn
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from io import BytesIO
import numpy as np
import cv2
import tensorflow as tf
import mediapipe as mp
import nest_asyncio
import os
nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
home = os.path.expanduser("~")
BASE = os.path.join(home, "Desktop", "GesturaAI_ANN")
# Load Model
model_api = tf.keras.models.load_model(os.path.join(BASE, "ann_model.h5"))
classes = np.load(os.path.join(BASE, "classes.npy"), allow_pickle=True)
mp_hands = mp.solutions.hands
hands_api = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    img = Image.open(BytesIO(contents)).convert('RGB')
    img = np.array(img)
    results = hands_api.process(img)
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            row_data = []
            for lm in hand_landmarks.landmark:
                row_data.extend([lm.x, lm.y])
            input_data = np.expand_dims(row_data, axis=0)
            preds = model_api.predict(input_data, verbose=0)
            idx = int(np.argmax(preds))
            conf = float(np.max(preds))
            return {"gesture": str(classes[idx]), "confidence": round(conf * 100, 2)}
    return {"gesture": "No hand detected", "confidence": 0.0}

def run_server():
    # Note: Changed port to 8080 to avoid conflict
    uvicorn.run(app, host="127.0.0.1", port=8080)

server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()

print("🚀 ANN Backend API is running on http://127.0.0.1:8080")

🚀 ANN Backend API is running on http://127.0.0.1:8080


INFO:     Started server process [1980]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)
C:\Users\SHAHID BALOCH\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49259 - "POST /predict HTTP/1.1" 200 OK
